# 🦜 LangChain Basics — Beginner's Notebook
### by - *Ashu Mishra*
### Reach me out on : https://www.linkedin.com/in/ashumish/
---

**What you'll learn today:**
1. How to talk to an AI model using Python
2. How to use Prompt Templates (reusable instructions for AI)
3. How to build a Chain (connect steps together like an assembly line)
4. How to get clean, structured output from AI

**Don't worry if you don't understand every line of code.**  
Focus on *what* each block does, not *how* Python works.

---

## 🔧 Step 1: Install the tools we need

Think of this like installing apps on your phone before using them.  
We are installing **LangChain** and **OpenAI** libraries.

> ⏱️ This may take 1-2 minutes. Wait for it to finish before moving ahead.

In [ ]:
!pip install langchain langchain-openai openai python-dotenv -q

## 🔑 Step 2: Set your OpenAI API Key

An **API Key** is like a password that lets us use OpenAI's AI model (GPT).

> 📌 Your instructor will share the API key during the session.  
> Replace `YOUR_OPENAI_API_KEY` below with the key shared with you.

In [ ]:
import os

# 👇 Paste your API key here (between the quotes)
os.environ['OPENAI_API_KEY'] = 'YOUR_OPENAI_API_KEY'

print('✅ API Key is set!')

---

## 🤖 Part 1: Talking to the AI Model

### What is an LLM?

**LLM = Large Language Model** — it's the brain behind tools like ChatGPT.

**Real-life analogy:**  
Imagine you have a very smart assistant sitting in a room.  
You pass a note through a slot → they read it → they write a reply → they pass it back.

That's exactly what we're doing with code!

```
You (Python code) → send a message → LLM reads it → LLM replies → you get the answer
```

In [ ]:
from langchain_openai import ChatOpenAI

# Create the AI model (we're using GPT-4o-mini — fast and affordable)
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7)

# Send a message to the AI and get a response
response = llm.invoke('What is LangChain? Explain in 2 simple sentences.')

# Print the response
print(response.content)

### 🌡️ What is `temperature`?

| Temperature | Behaviour | Use Case |
|-------------|-----------|----------|
| `0.0` | Very predictable, always same answer | Maths, factual questions |
| `0.7` | Creative but sensible | Most everyday tasks |
| `1.0` | Very creative, sometimes unpredictable | Storytelling, brainstorming |

Think of it like a **creativity dial** on your AI assistant.

---

## 📝 Part 2: Prompt Templates

### What is a Prompt Template?

**WhatsApp analogy:**  
Imagine you send the same WhatsApp message to 10 different people, but change only their name each time.

> *'Hey {name}, happy birthday!'*

You didn't rewrite the whole message 10 times. You had one message and changed **one word** each time.  
**That's a Prompt Template.**

In LangChain:
```
'Give me 3 tips for a {job_role}.'
```
Write once. Run for Product Manager, Data Analyst, HR Manager — same template, different input.

> **One-liner:** A Prompt Template is a message with a blank. You fill in the blank, AI does the rest.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Create a template with a placeholder {topic}
prompt_template = ChatPromptTemplate.from_messages([
    ('system', 'You are a friendly teacher who explains things in very simple terms.'),
    ('human', 'Explain {topic} in 3 bullet points. Use simple language and a real-life example.')
])

# Fill in the placeholder and see the final prompt
formatted_prompt = prompt_template.format_messages(topic='Machine Learning')
print('--- The prompt sent to AI ---')
for msg in formatted_prompt:
    print(f'[{msg.type.upper()}]: {msg.content}')

In [ ]:
# Now send this prompt to the AI and get a response
response = llm.invoke(formatted_prompt)
print(response.content)

---

## ⛓️ Part 3: Chains — The Assembly Line

### What is a Chain?

**Zomato analogy:**  
You place order → Restaurant cooks → Delivery picks up → Food arrives.  
Each step happens **in sequence**. No one asks — what next?

A **LangChain Chain** works the same way:
```
Prompt Template → LLM → Output Parser
```

The `|` pipe symbol connects each step:
```python
chain = prompt | llm | output_parser
```

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Step 1: Define the prompt template
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful career advisor.'),
    ('human', 'Give me 3 tips for someone working as a {job_role}. Keep it short and practical.')
])

# Step 2: Define the output parser
output_parser = StrOutputParser()

# Step 3: Connect them into a chain using the pipe | operator
chain = prompt | llm | output_parser

# Step 4: Run the chain!
result = chain.invoke({'job_role': 'Product Manager'})
print(result)

In [ ]:
# The beauty of a chain — reuse it with different inputs!
result2 = chain.invoke({'job_role': 'Data Analyst'})
print(result2)

---

## 📦 Part 4: Output Parsers — Getting Clean Output

### What is an Output Parser?

AI models return text. But sometimes you need a **list**, or a **specific format**.

**Meeting availability analogy:**  
You ask 10 colleagues for their availability. Each replies differently — messy, unstructured.  
Your assistant converts all those replies into one clean table.  
**That's an Output Parser.**

> **One-liner:** An Output Parser is your assistant who takes a messy reply and converts it into something clean, structured, and usable.

## 📋 Case Study: Making AI Output Usable by a System

### The Problem

You are building an AI agent for a product team.
The agent needs to:
1. Get a list of AI tools from the AI
2. Pass that list to the next step — save to database, show in dashboard, or trigger another workflow

Simple enough. Let's try it.

### 🔴 Step 1 — Primitive Solution (No Parser)

Let's ask the AI for the list the straightforward way.  
Run this and look carefully at the output.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),
    ('human', 'List 5 popular AI tools used in product management.')
])

chain = prompt | llm

result = chain.invoke({})

print('Type of result:', type(result))
print('\nOutput:')
print(result.content)

### ❌ Why this output is a problem

Look at what we got:

- `Type of result` → `AIMessage` — one big blob of text. Not a list.
- The AI replied like a human — full sentences, explanations, commas everywhere.

Now ask yourself:

| Question | Answer |
|---|---|
| Can I save each tool separately to a database? | ❌ No |
| Can the next agent step loop through this? | ❌ No |
| Will I get the same format every time I run it? | ❌ No |
| Can a dashboard read this automatically? | ❌ No |

**The AI talked like a human. But machines need to talk to other machines.**  
We need to fix this.

### 🟢 Step 2 — Introducing the Output Parser

We are going to make two small changes to the code:

1. Add a parser — `CommaSeparatedListOutputParser()`
2. Ask the parser to give the AI format instructions before it replies

That's it. Same question. Same AI. Watch what changes.

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# Change 1 — create the parser
parser = CommaSeparatedListOutputParser()

# Change 2 — get format instructions from the parser
format_instructions = parser.get_format_instructions()

print('What the parser tells the AI:')
print(format_instructions)

> 👆 Read that format instruction carefully.  
> The parser is telling the AI:  
> *'Don't give me a paragraph. Give me values separated by commas. Like this: foo, bar, baz'*  
>  
> Now let's pass this instruction into the prompt and run the chain.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),
    ('human', 'List 5 popular AI tools used in product management.\n{format_instructions}')
])

# Change 3 — add parser at the end of the chain
chain = prompt | llm | parser

result = chain.invoke({'format_instructions': format_instructions})

print('Type of result:', type(result))
print('\nAI Tools:')
for i, tool in enumerate(result, 1):
    print(f'  {i}. {tool}')

### 🔍 What Changed — Exactly

Only 3 lines were added. Nothing else touched.

| What we added | Why |
|---|---|
| `CommaSeparatedListOutputParser()` | Created the parser |
| `parser.get_format_instructions()` | Told the AI how to format its reply |
| `| parser` at end of chain | Told the chain to clean the output |

### ⚙️ How That Change Affected the Output

The parser did two jobs — one BEFORE the AI replied, one AFTER.

**Job 1 — BEFORE the AI replied:**  
The parser gave the AI a clear instruction: *'Reply in comma separated format only. No sentences. No explanation.'*  
The AI now knew exactly what to return.

**Job 2 — AFTER the AI replied:**  
The parser took the AI's clean comma separated reply and split it into a proper Python list.

| | Without Parser | With Parser |
|---|---|---|
| Type | `AIMessage` — blob of text | `list` — usable data |
| Loopable? | ❌ No | ✅ Yes |
| Saveable to database? | ❌ No | ✅ Yes |
| Consistent every run? | ❌ No | ✅ Yes |
| System digestible? | ❌ No | ✅ Yes |

**One parser. Three lines. Completely different result.**

---

## 🎯 Quick Recap — What We Learned

| Concept | What it does | Real-life analogy |
|---------|-------------|-------------------|
| **LLM** | The AI brain that generates text | Smart assistant in a room |
| **Prompt Template** | Reusable instructions with placeholders | WhatsApp birthday message |
| **Chain** | Connects steps in sequence | Zomato delivery pipeline |
| **Output Parser** | Converts AI text to clean format | Meeting availability table |

The magic formula:
```
chain = prompt_template | llm | output_parser
result = chain.invoke({'your_variable': 'your_value'})
```

---

## ✏️ Exercise — Your Turn!

### Change only these 3 things:

| # | What to change | Hint |
|---|---|---|
| 1 | System message | Tell AI it is an HR assistant |
| 2 | Human message | Ask for top 5 skills for {job_role} |
| 3 | Your job role | Put your own role here |

Do not touch anything else.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()
format_instructions = parser.get_format_instructions()

# 👇 Change 3: put your job role here
job_role = 'Marketing Manager'   # 👈 change this to YOUR role

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),          # 👈 Change 1
    ('human', 'List 5 popular AI tools.\n{format_instructions}')  # 👈 Change 2
])

chain = prompt | llm | parser

# Do not touch this
result = chain.invoke({'format_instructions': format_instructions})

print(f'Results for: {job_role}')
print('\nAI Tools:')  # 👈 rename this label too if you like
for i, item in enumerate(result, 1):
    print(f'  {i}. {item}')

### 🌟 Bonus Challenge

Copy your working code into the cell below.  
Change the `job_role` value 3 times and run:

- `'Product Manager'`
- `'Data Analyst'`
- `'Software Engineer'`

Do the skills change? Are any skills common across all 3?

---

### 🎉 You built your first LangChain app!

**What's coming next:**
- 🧠 **Memory** — make AI remember previous conversations
- 📄 **RAG** — make AI answer questions from your own documents
- 🤖 **Agents** — give AI tools to search the web, run calculations, and more!